<a href="https://colab.research.google.com/github/hamzafarooq/multi-agent-course/blob/main/modules/Module_3_Production_Agentic_RAG_AI_Systems/001.%20Agentic%20Router.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Dive Agentic Retrieval Augmented Generation

An Agentic RAG is required when we use reasoning to determine which action(s) to take and in which order to take them. Essentially we use agents instead of a LLM directly to accomplish a set of tasks which requires planning, multi step reasoning, tool use and/or learning over time. Agents give us agency!

Agency : The ability to take action or to choose what action to take

In the context of RAG, we can plug in agents to enhance the reasoning prior to selection of RAG pipelines, within a RAG pipeline for retrieval or reranking and finally for synthesising before we send out the response. This improves RAG to a large extent by automating complex workflows and decisions that are required for a non trivial RAG use case.

### Purpose of this Agentic RAG
This notebook presents a practical implementation of Agentic Retrieval-Augmented Generation (RAG)—a system where decision-making and tool selection are delegated to an intelligent agent before executing a response. Rather than passing every query through a static RAG pipeline, this system introduces agency—the ability to choose the best course of action depending on the nature of the query.

At the heart of this implementation is a router prompt, which classifies user queries into one of three categories:

- OpenAI documentation: Queries related to tools, APIs, or usage guidelines for OpenAI models
- 10-K financial reports: Questions requiring retrieval from company filings or financial datasets
- Live Internet search: Broader, current, or comparative queries that need web access

Once the query is classified, the system invokes a corresponding route handler:

- For OpenAI and 10-K queries, it retrieves relevant context from a vector database (Qdrant) using text embeddings, then applies a RAG-based response generator.
- For Internet queries, it fetches real-time information using a web-access API (ARES).

This approach is an example of Agentic RAG, where reasoning precedes retrieval and generation. By plugging in agents before and within the RAG pipeline, we make the system smarter and more adaptive. This allows us to:

- Automatically choose the right retrieval method based on context
- Combine structured knowledge with real-time search
- Scale RAG beyond trivial use cases by integrating multi-step decision logic

Importantly, no external agentic frameworks are used—this is a ground-up implementation that demonstrates how to build a lightweight but intelligent agentic system using only a language model, prompt engineering, and retrieval tools.

## Setup and Dependencies

In [1]:
# Install the necessary libraries
!pip install openai
!pip install qdrant_client
!pip install transformers==4.48.0

zsh:1: command not found: pip


zsh:1: command not found: pip


zsh:1: command not found: pip


In [2]:
# Import basic libraries
import requests             # Used for making HTTP requests (e.g., calling ARES API for live internet queries)
import json                 # For parsing and structuring JSON data (especially OpenAI and routing responses)

# Credentials — Colab Secrets when on Colab, a local .env otherwise
try:
    from google.colab import userdata          # Colab: keys live in the 🔑 Secrets panel
    IN_COLAB = True
except ImportError:                            # Local Jupyter: keys live in .env
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv())
    IN_COLAB = False

    class userdata:                            # same .get() call works in both places
        @staticmethod
        def get(name):
            import os
            return os.getenv(name) or os.getenv(name.lower()) or os.getenv(
                name.replace("SERP_API_KEY", "SERPAPI_KEY"))

# OS operations
import os                   # Useful for accessing environment variables and managing paths

# OpenAI API client
from openai import OpenAI   # Official OpenAI client library to interface with GPT models for routing and generation

# Text processing
import re                   # Regular expressions for cleaning or preprocessing inputs (if needed)

# Optional visualization (for analysis/debugging purposes)
import matplotlib.pyplot as plt       # For displaying charts or visual debug outputs (e.g., embeddings visualizations)
import matplotlib.image as mpimg      # For loading/displaying images if needed (rare in RAG, but helpful in demos)

# Embedding models (used for text vectorization during retrieval)
from transformers import AutoTokenizer, AutoModel  # For loading custom transformer models if not using OpenAI embeddings

from qdrant_client import models

import qdrant_client
import asyncio
import nest_asyncio # Import nest_asyncio
nest_asyncio.apply() # Apply nest_asyncio to allow nested event loops
# # Vector database client
# from qdrant_client import QdrantClient   # Qdrant is used as the vector store to retrieve documents based on similarity

## 1. Defining the Internet Tool

First, we will define a tool function that enables our system to answer queries requiring real-time, internet-based information. Not all questions can be answered using static documents like OpenAI docs or financial filings—sometimes users ask about current trends, comparisons, or live updates.

To handle this, we introduce a live search capability using the **SerpApi**.

### What is SerpApi?  
SerpApi is a Google Search API that allows you to:

- Search the internet in real time using Google.
- Get structured results including answer boxes, organic results, and snippets.

This is particularly useful for questions about:

- Current events (e.g., *"Latest AI tools in 2025"*),
- Tech comparisons (e.g., *"Gemini vs GPT-4"*),
- General knowledge outside internal datasets.

Please generate the API key [here](https://serpapi.com)


In [3]:
#loads serp api key from colab secrets
serp_api_key=userdata.get('SERP_API_KEY')

In [4]:
import requests  # For sending HTTP requests to the SerpApi

def get_internet_content(user_query: str, action: str):
    """
    Fetches a response from the internet using SerpApi based on the user's query.

    This function serves as the tool invoked when the router classifies a query
    as requiring real-time information beyond internal datasets—i.e., "INTERNET_QUERY".
    It sends the query to SerpApi (Google Search) and returns structured results.

    Args:
        user_query (str): The user's question that needs a live answer.
        action (str): Route type (always expected to be "INTERNET_QUERY").

    Returns:
        str: Response text from live Google search results or an error message.
    """
    print("Getting your response from the internet 🌐 ...")

    params = {
        "q": user_query,
        "api_key": serp_api_key,
        "engine": "google",
        "num": 5,
    }

    try:
        response = requests.get("https://serpapi.com/search.json", params=params)
        response.raise_for_status()
        data = response.json()

        parts = []

        # Answer box — Google's highlighted direct answer (most relevant)
        answer_box = data.get("answer_box", {})
        if answer_box.get("answer"):
            parts.append(f"[Direct Answer] {answer_box['answer']}")
        elif answer_box.get("snippet"):
            parts.append(f"[Direct Answer] {answer_box['snippet']}")

        # Top organic results — titles + snippets
        for i, result in enumerate(data.get("organic_results", [])[:5], start=1):
            title = result.get("title", "")
            snippet = result.get("snippet", "")
            link = result.get("link", "")
            if snippet:
                parts.append(f"[{i}] {title}\n    {snippet}\n    Source: {link}")

        if not parts:
            return "No results found."

        return "\n\n".join(parts)

    # Handle HTTP-level errors (e.g., 400s or 500s)
    except requests.exceptions.HTTPError as http_err:
        return f"HTTP error occurred: {http_err}"

    # Handle general connection, timeout, or request formatting issues
    except requests.exceptions.RequestException as req_err:
        return f"Request error occurred: {req_err}"

    # Catch-all for any unexpected failure
    except Exception as err:
        return f"An unexpected error occurred: {err}"


In [5]:
print(get_internet_content("Tell me about best travel destinations in 2026?","INTERNET_QUERY")) #run internet function to test results

Getting your response from the internet 🌐 ...
[1] My Top 26 Best Travel Destinations For 2026
    My Top 26 Best Travel Destinations For 2026 · Kyrgyzstan · Kazakhstan · Uzbekistan · Tajikistan · Pakistan · Nepal · Mexico · Colombia. Colombia ...
    Source: https://thepartyingtraveler.com/2022/01/03/26-best-travel-destinations-to-visit-in-2026/

[2] 2026 Travel Destinations: 52 Places to Go This Year
    52 Places to Go in 2026. Our list for the new year features an eclipse, a revolution and a tiger reserve. What's on yours?
    Source: https://www.nytimes.com/interactive/2026/travel/places-to-travel-destinations-2026.html

[3] where are people planning to travel in 2026? : r/Shoestring
    Where are people planning to travel in 2026? So far I have vague plans for Italy and Chile/Argentina.
    Source: https://www.reddit.com/r/Shoestring/comments/1p8d3za/where_are_people_planning_to_travel_in_2026/

[4] 30 World's Best Places to Visit for 2026
    World's Best Places to Visit for 2026

## 2. Router Query Function — Giving the Agent Its Brain

In this step, we will define the router function, which plays a critical role in our Agentic RAG system.

### What is a Router?

A router is like the decision-making brain of our assistant.

Before trying to answer a user's question, the system first needs to figure out:

> “Where should I go to find the right answer?”

To make this decision, we use the OpenAI GPT model. We provide it with a detailed system prompt that explains how to classify the user's question into one of these categories:

- **OPENAI_QUERY** → Questions about OpenAI tools, APIs, models, or documentation.
- **10K_DOCUMENT_QUERY** → Questions about companies, financial filings, or analysis based on 10-K reports.
- **INTERNET_QUERY** → Anything else that likely requires real-time or general web information.

### What does the function do?

- Sends the user's question to the OpenAI API.
- Receives a JSON response containing:
  - `action`: The category the query belongs to.
  - `reason`: A short explanation for the decision.
  - `answer`: (Optional) A quick response if it’s simple enough (left blank for internet queries).
- Parses the response and returns it as a Python dictionary.

### Why is this important?

This router gives the system agency—the ability to decide which knowledge source to use. It’s what makes this pipeline agentic, not just static.

Without the router, every query would follow the same path. With it, we can:

- Dynamically switch between tools and data sources.
- Handle different types of user questions intelligently.
- Avoid wasting resources on unnecessary steps.


## Query Routing Workflow

The diagram below shows the full decision flow — from receiving a user query to returning a final response.

```
                        ┌─────────────────────┐
                        │     User Query      │
                        └──────────┬──────────┘
                                   │
                                   ▼
                    ┌──────────────────────────────┐
                    │      Router LLM (GPT-4o)     │
                    │         route_query()         │
                    │                              │
                    │  Reads the query and decides │
                    │  which data source to use    │
                    └──────────────┬───────────────┘
                                   │
           ┌───────────────────────┼───────────────────────┐
           │                       │                       │
           ▼                       ▼                       ▼
┌─────────────────────┐ ┌─────────────────────┐ ┌─────────────────────┐
│    OPENAI_QUERY     │ │ 10K_DOCUMENT_QUERY  │ │   INTERNET_QUERY    │
│                     │ │                     │ │                     │
│ e.g. "What are      │ │ e.g. "What was      │ │ e.g. "Best LLMs     │
│  OpenAI Agents?"    │ │  Uber's revenue?"   │ │  in 2026?"          │
└──────────┬──────────┘ └──────────┬──────────┘ └──────────┬──────────┘
           │                       │                        │
           ▼                       ▼                        ▼
┌─────────────────────┐ ┌─────────────────────┐  ┌──────────────────────┐
│   Embed Query       │ │   Embed Query        │  │      SerpApi         │
│  (Nomic Model)      │ │  (Nomic Model)       │  │  get_internet_       │
│                     │ │                      │  │  content()           │
│  get_text_          │ │  get_text_           │  │                      │
│  embeddings()       │ │  embeddings()        │  │  Live Google search  │
└──────────┬──────────┘ └──────────┬──────────┘  └──────────┬───────────┘
           │                       │                          │
           ▼                       ▼                          │
┌─────────────────────┐ ┌─────────────────────┐              │
│  Qdrant Vector DB   │ │  Qdrant Vector DB   │              │
│  Collection:        │ │  Collection:         │              │
│  "opnai_data"       │ │  "10k_data"          │              │
│                     │ │                      │              │
│  Retrieve top-3     │ │  Retrieve top-3      │              │
│  similar chunks     │ │  similar chunks      │              │
└──────────┬──────────┘ └──────────┬──────────┘              │
           │                       │                          │
           └───────────┬───────────┘                          │
                       ▼                                      │
           ┌───────────────────────┐                          │
           │    RAG Response       │                          │
           │    Generator          │                          │
           │  rag_formatted_       │                          │
           │  response()           │                          │
           │                       │                          │
           │  GPT-4 synthesizes    │                          │
           │  answer from context  │                          │
           │  + adds citations     │                          │
           └───────────┬───────────┘                          │
                       │                                      │
                       └──────────────────┬───────────────────┘
                                          ▼
                             ┌────────────────────────┐
                             │     Final Response     │
                             │       to User          │
                             └────────────────────────┘
```

### Key decision points at a glance

| Route | Trigger | Retrieval Method | Response Generator |
|---|---|---|---|
| `OPENAI_QUERY` | OpenAI docs, APIs, Agents | Qdrant `opnai_data` (top-3 chunks) | `rag_formatted_response()` via GPT-4 |
| `10K_DOCUMENT_QUERY` | Financial filings, company revenue | Qdrant `10k_data` (top-3 chunks) | `rag_formatted_response()` via GPT-4 |
| `INTERNET_QUERY` | Anything else / real-time info | SerpApi live Google search | Raw search result snippets |

> **Note:** Both vector-based routes share the same embedding model (`nomic-embed-text-v1.5`) and RAG generator — only the Qdrant collection changes. The router's JSON output (`action` field) is the single decision variable that drives the entire flow.


In [6]:
# Securely retrieve the OpenAI API key from Colab's user data store
# This avoids hardcoding sensitive credentials directly in the notebook
openai_api_key = userdata.get('OPENAI_API_KEY')

# Initialize the OpenAI client with the retrieved API key
# This client will be used for:
# - Query classification via the router prompt
# - Potentially generating responses from retrieved context
openaiclient = OpenAI(api_key=openai_api_key)


In [7]:
from openai import OpenAIError

def route_query(user_query: str):
    router_system_prompt =f"""
    As a professional query router, your objective is to correctly classify user input into one of three categories based on the source most relevant for answering the query:
    1. "OPENAI_QUERY": If the user's query appears to be answerable using information from OpenAI's official documentation about Agents, tools, models, APIs, or services (e.g., guardrails, agents, what is an agent, embeddings, moderation API, usage guidelines).
    2. "10K_DOCUMENT_QUERY": If the user's query pertains to a collection of documents from the 10k annual reports, datasets, or other structured documents, typically for research, analysis, or financial content.
    3. "INTERNET_QUERY": If the query is neither related to OpenAI nor the 10k documents specifically, or if the information might require a broader search (e.g., news, trends, tools outside these platforms), route it here.

    Your decision should be made by assessing the domain of the query.

    Always respond in this valid JSON format:
    {{
        "action": "OPENAI_QUERY" or "10K_DOCUMENT_QUERY" or "INTERNET_QUERY",
        "reason": "brief justification",
        "answer": "AT MAX 5 words answer. Leave empty if INTERNET_QUERY"
    }}

    EXAMPLES:

    - User: "How to fine-tune GPT-3?"
    Response:
    {{
        "action": "OPENAI_QUERY",
        "reason": "Fine-tuning is OpenAI-specific",
        "answer": "Use fine-tuning API"
    }}

    - User: "Where can I find the latest financial reports for the last 10 years?"
    Response:
    {{
        "action": "10K_DOCUMENT_QUERY",
        "reason": "Query related to annual reports",
        "answer": "Access through document database"
    }}

    - User: "Top leadership styles in 2024"
    Response:
    {{
        "action": "INTERNET_QUERY",
        "reason": "Needs current leadership trends",
        "answer": ""
    }}

    - User: "What's the difference between ChatGPT and Claude?"
    Response:
    {{
        "action": "INTERNET_QUERY",
        "reason": "Cross-comparison of different providers",
        "answer": ""
    }}

    Strictly follow this format for every query, and never deviate.
    User: {user_query}
    """

    try:
        # Query the GPT-4 model with the router prompt and user input
        response = openaiclient.chat.completions.create(
            model="gpt-5.6-luna",
            messages=[{"role": "system", "content": router_system_prompt}]
        )

        # Extract and parse the model's JSON response
        task_response = response.choices[0].message.content
        json_match = re.search(r"\{.*\}", task_response, re.DOTALL)
        json_text = json_match.group()
        parsed_response = json.loads(json_text)
        return parsed_response

    # Handle OpenAI API errors (e.g., rate limits, authentication)
    except OpenAIError as api_err:
        return {
            "action": "INTERNET_QUERY",
            "reason": f"OpenAI API error: {api_err}",
            "answer": ""
        }

    # Handle case where model response isn't valid JSON
    except json.JSONDecodeError as json_err:
        return {
            "action": "INTERNET_QUERY",
            "reason": f"JSON parsing error: {json_err}",
            "answer": ""
        }

    # Catch-all for any other unforeseen issues
    except Exception as err:
        return {
            "action": "INTERNET_QUERY",
            "reason": f"Unexpected error: {err}",
            "answer": ""
        }

In [8]:
route_query("what is the revenue of uber in 2021?")


{'action': '10K_DOCUMENT_QUERY',
 'reason': "Asks for Uber's annual financial revenue",
 'answer': "Uber's 2021 revenue"}

In [9]:
route_query("what is an AI Agent?")

{'action': 'OPENAI_QUERY',
 'reason': "AI agents are covered in OpenAI's official documentation",
 'answer': 'Software that performs tasks'}

## 3. Setting Up Qdrant Vector Database for Agentic RAG
In this step, we are connecting our agent to a pre-built vector database using Qdrant—a tool used to store and search document embeddings (numerical representations of text).

What Are We Doing?
We are loading an existing Qdrant database that was downloaded from a GitHub repository. This database already contains:

- Vectorized OpenAI documentation
- Vectorized 10-K financial filings

By loading this saved data:

- We save time (no need to re-embed the documents)
- We enable fast similarity search to retrieve relevant text chunks

This setup allows our system to perform semantic search, meaning it can understand the meaning of the user query and match it with the most relevant pieces of information stored in the database.


### Why This Matters in Agentic RAG
Once the router decides that the query should go to the OpenAI docs or the 10-K reports, our system uses Qdrant to:

- Search for the most relevant pieces of text
- Pass those to the model to generate a grounded answer

So, this step is essential to support retrieval-augmented generation (RAG) within our agentic flow.

#Data Sources:

**10K Database: Lyft 2024 & Uber 2021 SEC filings**

**OpenAI Docs: Official OpenAI documentation about Agents**

For lecture demo purposes, the vecitr database has already been created and hosted on Github which we will clone here. In order to create your own embeddings, the notebook and data will be hosted and shared on github

In [10]:
# Colab only — wipe a previous clone if you need a clean copy
if IN_COLAB:
    !rm -rf /content/multi-agent-course

In [11]:
# The prebuilt Qdrant collections (10-K + OpenAI docs) ship with the repo.
# On Colab we clone to get them; locally you already have them.
if IN_COLAB:
    !git clone https://github.com/hamzafarooq/multi-agent-course.git

In [12]:
# 🗄️ Initializing Qdrant client with the local path to the vector database
# Prebuilt collections (10-K and OpenAI docs) — cloned on Colab, already present locally.
import os

_MODULE = "modules/Module_3_Production_Agentic_RAG_AI_Systems"
if IN_COLAB:
    QDRANT_PATH = f"/content/multi-agent-course/{_MODULE}/Agentic_RAG/qdrant_data"
else:
    # the notebook lives in the module folder, so the data sits right next to it
    QDRANT_PATH = os.path.join(os.getcwd(), "Agentic_RAG", "qdrant_data")

print("Qdrant path:", QDRANT_PATH)
client = qdrant_client.AsyncQdrantClient(path=QDRANT_PATH)

Qdrant path: /Users/kurtlozier/Learning/Hamza_Cohort_02_Forward_Deployed_Engineering_Bootcamp/Assignment_2_Python_Code_n_ARGUS/part1_agentic_router/Agentic_RAG/qdrant_data


## 4. Building the Retriever and RAG for Vector Databases
In this section, we build the core logic that allows our agent to find relevant documents and generate grounded answers using them.

###Step 1: Import the Embedding Model
We start by importing the nomic-ai/nomic-embed-text-v1.5 model from Hugging Face. This model is used to convert any text (such as a user query) into a dense vector, known as an embedding. These embeddings capture the semantic meaning of text, allowing us to later compare and retrieve similar documents.


In [13]:
# Load the tokenizer and embedding model from Hugging Face
# This model converts raw text into dense vector representations (embeddings)
# Used for similarity search in Qdrant during document retrieval
text_tokenizer = AutoTokenizer.from_pretrained("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)
text_model = AutoModel.from_pretrained("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)

def get_text_embeddings(text):
    """
    Converts input text into a dense embedding using the Nomic embedding model.
    These embeddings are used to query Qdrant for semantically relevant document chunks.

    Args:
        text (str): The input text or query from the user.

    Returns:
        np.ndarray: A fixed-size vector representing the semantic meaning of the input.
    """
    # Tokenize and prepare input for the model
    inputs = text_tokenizer(text, return_tensors="pt", padding=True, truncation=True)

    # Forward pass to get model outputs
    outputs = text_model(**inputs)

    # Take the mean across all token embeddings to get a single vector (pooled representation)
    embeddings = outputs.last_hidden_state.mean(dim=1)

    # Convert to NumPy array and detach from computation graph
    return embeddings[0].detach().numpy()

# Example usage: Generate and preview the embedding of a test sentence
text = "This is a test sentence."
embeddings = get_text_embeddings(text)
print(embeddings[:5])  # Print first 5 dimensions for inspection


<All keys matched successfully>


[ 1.2799684   0.4015834  -3.5162644  -0.39813134  1.5919116 ]


### Step 2: Define the Embedding Function
We then define a function get_text_embeddings() which:

- Tokenizes the input text
- Runs it through the model
- Computes the average of all token embeddings
- Returns a single vector that represents the full sentence

This vector will be used to query Qdrant to find the most relevant document chunks based on similarity.

In [14]:
def rag_formatted_response(user_query: str, context: list):
    """
    Generate a response to the user query using the provided context,
    with article references formatted as [1][2], etc.

    This function performs the final step in the RAG pipeline—synthesizing an answer
    from retrieved document chunks (context). It prompts the model to generate a
    grounded response, explicitly citing sources using a reference format.

    Args:
        user_query (str): The user's original question.
        context (list): List of text chunks retrieved from Qdrant (10-K or OpenAI docs).

    Returns:
        str: A generated response grounded in the retrieved context, with numbered citations.
    """

    # Construct a RAG prompt that includes both:
    # 1. The user's query
    # 2. The supporting context documents
    # The prompt instructs the model to answer using only the provided context,
    # and to include citations like [1], [2], etc. based on chunk IDs or order.
    rag_prompt = f"""
       Based on the given context, answer the user query: {user_query}\nContext:\n{context}
       and employ references to the ID of articles provided [ID], ensuring their relevance to the query.
       The referencing should always be in the format of [1][2]... etc. </instructions>
    """

    #  Call GPT-5.6-Luna to generate the response using the RAG-style prompt
    response = openaiclient.chat.completions.create(
        model="gpt-5.6-luna",
        messages=[
            {"role": "system", "content": rag_prompt},
        ]
    )

    # Return the model's generated answer
    return response.choices[0].message.content


### Step 3: Define the RAG Response Generator
After retrieving relevant text chunks from Qdrant, we use the rag_formatted_response() function to generate a final answer. This function:

- Takes the user query and the retrieved document chunks
- Builds a prompt that asks the language model (GPT-5.6-Luna) to answer the question using only the provided context
- Instructs the model to include references like [1], [2] for traceability

This ensures the output is not only informative but also grounded in actual retrieved data.

Together, these two functions lay the foundation for combining retrieval (from vector DB) and generation (from LLM) — the two pillars of a RAG system.



In [15]:
async def retrieve_and_response(user_query: str, action: str):
    """
    Retrieves relevant text chunks from the appropriate Qdrant collection
    based on the query type, then generates a response using RAG.

    This function powers the retrieval and response generation pipeline
    for queries that are classified as either OPENAI-related or 10-K related.
    It uses semantic search to fetch relevant context from a Qdrant vector store
    and then generates a response using that context via a RAG prompt.

    Args:
        user_query (str): The user's input question.
        action (str): The classification label from the router (e.g., "OPENAI_QUERY", "10K_DOCUMENT_QUERY").

    Returns:
        str: A model-generated response grounded in retrieved documents, or an error message.
    """

    # Define mapping of routing labels to their respective Qdrant collections
    collections = {
        "OPENAI_QUERY": "opnai_data",           # Collection of OpenAI documentation embeddings
        "10K_DOCUMENT_QUERY": "10k_data"        # Collection of 10-K financial document embeddings
    }

    try:
        # Ensure that the provided action is valid
        if action not in collections:
            return "Invalid action type for retrieval."

        # Step 1: Convert the user query into a dense vector (embedding)
        try:
            query = get_text_embeddings(user_query)
        except Exception as embed_err:
            return f"Embedding error: {embed_err}"  # Fail early if embedding fails

        # Step 2: Retrieve top-matching chunks from the relevant Qdrant collection
        try:
            text_hits = await client.query_points(
                collection_name=collections[action],  # Choose the right collection based on routing
                query=query,                          # The embedding of the user's query
                limit=3                               # Fetch top 3 relevant chunks
            )
        except Exception as qdrant_err:
            return f"Vector DB query error: {qdrant_err}"  # Handle Qdrant access issues

        # Extract the raw content from the retrieved vector hits
        contents = [point.payload['content'] for point in text_hits.points]

        # If no relevant content is found, return early
        if not contents:
            return "No relevant content found in the database."

        # Step 3: Pass the retrieved context to the RAG model to generate a response
        try:
            response = rag_formatted_response(user_query, contents)
            return response
        except Exception as rag_err:
            return f"RAG response error: {rag_err}"  # Handle generation failures

    # Catch any unforeseen errors in the overall process
    except Exception as err:
        return f"Unexpected error: {err}"


# 5. Putting It All Together: Running the Agentic RAG
In this final step, we combine everything into a single function that controls the entire Agentic RAG workflow. The agentic_rag() function acts as the main orchestrator of the system.

Here’s what it does:

- Prints the user's query for reference.
- Uses the router function (powered by GPT) to decide which type of data source to use:
  - OpenAI documentation
  - 10-K financial reports
- Internet search
- Calls the correct function based on the route:
- If it’s an OpenAI or 10-K query, it retrieves data from Qdrant and generates a RAG response.
- If it’s an Internet query, it uses the ARES API to fetch live information.
- Displays the final response, neatly formatted in the console.

This step brings the agentic loop full circle—from understanding the question, reasoning about where to search, to finally responding with the best possible answer.

In [16]:
# Dictionary that maps the route labels (decided by the router) to their respective functions
# Each type of query is handled differently:
# - OPENAI_QUERY and 10K_DOCUMENT_QUERY use document retrieval + RAG
# - INTERNET_QUERY uses a web search API
routes = {
    "OPENAI_QUERY": retrieve_and_response,
    "10K_DOCUMENT_QUERY": retrieve_and_response,
    "INTERNET_QUERY": get_internet_content,
}

def agentic_rag(user_query: str):
    """
    Main function that runs the full Agentic RAG system.

    This function takes a user's question, decides what type of query it is (OpenAI-related,
    financial document-related, or general internet), and then calls the right function
    to handle it. Finally, it prints out the full conversation and response.

    Args:
        user_query (str): The user's input question.

    Returns:
        None (It just prints the result nicely to the console)
    """

    #  Terminal color codes to make the printed output easier to read and visually structured
    CYAN = "\033[96m"
    GREY = "\033[90m"
    BOLD = "\033[1m"
    RESET = "\033[0m"

    try:
        # Step 1: Print the user's original question to the console
        print(f"{BOLD}{CYAN}👤 User Query:{RESET} {user_query}\n")

        # Step 2: Use the router (powered by GPT) to decide which route the query belongs to
        try:
            response = route_query(user_query)
        except Exception as route_err:
            # If something goes wrong while classifying the query, show an error message
            print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n")
            print(f"Routing error: {route_err}\n")
            return

        # Extract the routing decision and the reason behind it
        action = response.get("action")  # e.g., "OPENAI_QUERY"
        reason = response.get("reason")  # e.g., "Related to OpenAI tools"

        # Step 3: Show the selected route and why it was chosen
        print(f"{GREY}📍 Selected Route: {action}")
        print(f"📝 Reason: {reason}")
        print(f"⚙️ Processing query...{RESET}\n")

        # Step 4: Call the correct function depending on the route (retrieval or web search)
        try:
            route_function = routes.get(action)  # Find the function to use for this route
            if route_function:
                if action in ["OPENAI_QUERY", "10K_DOCUMENT_QUERY"]:
                    # Use asyncio.run for async functions, nest_asyncio will handle nested loops
                    result = asyncio.run(route_function(user_query, action))
                else:
                    # Otherwise, call it directly (e.g., get_internet_content is synchronous)
                    result = route_function(user_query, action)
            else:
                result = f"Unsupported action: {action}"  # Catch unknown routing types
        except Exception as exec_err:
            result = f"Execution error: {exec_err}"  # Handle failure in the chosen route function

        # Step 5: Print the final response to the user
        print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n")
        print(f"{result}\n")

    except Exception as err:
        # Catch-all for any unexpected errors in the overall logic
        print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n")
        print(f"Unexpected error occurred: {err}\n")


In [17]:
agentic_rag("what was uber revenue in 2021?")

👤 User Query: what was uber revenue in 2021?



📍 Selected Route: 10K_DOCUMENT_QUERY
📝 Reason: Asks for Uber's annual revenue from a specific reporting year.
⚙️ Processing query...



🤖 BOT RESPONSE:

Uber’s revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**). [1]



In [18]:
agentic_rag("what was lyft revenue in 2022?")

👤 User Query: what was lyft revenue in 2022?



📍 Selected Route: 10K_DOCUMENT_QUERY
📝 Reason: Asks for Lyft's annual revenue from a 10-K report
⚙️ Processing query...



🤖 BOT RESPONSE:

Lyft’s revenue in 2022 was **$4.095 billion** (or **$4,095,135 thousand**), an increase of 28% from 2021. [1]



In [19]:
agentic_rag("List me down new LLMs in 2025")

👤 User Query: List me down new LLMs in 2025



📍 Selected Route: INTERNET_QUERY
📝 Reason: Requires current information about newly released language models
⚙️ Processing query...

Getting your response from the internet 🌐 ...
🤖 BOT RESPONSE:

[1] Top LLMs To Use in 2026: Our Best Picks
    Released on February 17, 2025, Grok-3 was designed to compete with the very best: GPT-4o, Gemini 2.5, and Claude 3.7. What stood out to me most ...
    Source: https://www.splunk.com/en_us/blog/learn/llms-best-to-use.html

[2] The 10 Best Large Language Models (LLMs) in 2026
    The best LLMs in 2025, Top LLM providers—including OpenAI, Anthropic, Google DeepMind, Meta, DeepSeek, xAI, and Mistral—each. DeepSeek R1 and Gemini 2.5 Pro ...
    Source: https://botpress.com/blog/best-large-language-models

[3] 2025 is over. What were the best AI model releases this ...
    2025 felt like three AI years compressed into one. Frontier LLMs went insane on reasoning, open‑source finally became “good enough” for a ton ...
    Source: https://www.reddit.com/

In [20]:
agentic_rag("how to work with chat completions?")

👤 User Query: how to work with chat completions?



📍 Selected Route: OPENAI_QUERY
📝 Reason: Chat Completions is an OpenAI API
⚙️ Processing query...



🤖 BOT RESPONSE:

Chat Completions lets you send a sequence of messages to a model and receive an assistant response. The basic workflow is:

1. Define system instructions.
2. Add the user’s message.
3. Call the model.
4. Read the assistant’s response.
5. Preserve the conversation history for follow-up turns.

### Basic example

```python
from openai import OpenAI

client = OpenAI()

messages = [
    {
        "role": "system",
        "content": "You are a helpful customer-support assistant."
    },
    {
        "role": "user",
        "content": "Where is my order?"
    }
]

completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages
)

answer = completion.choices[0].message.content
print(answer)
```

### Multi-turn conversations

Include previous messages in later requests:

```python
messages.append({
    "role": "assistant",
    "content": answer
})

messages.append({
    "role": "user",
    "content": "The order number is 12345."
})

completion = c

In [21]:
agentic_rag("best ways to build Agents")

👤 User Query: best ways to build Agents



📍 Selected Route: OPENAI_QUERY
📝 Reason: Agent development is covered by OpenAI documentation
⚙️ Processing query...



🤖 BOT RESPONSE:

## Best ways to build effective agents

1. **Choose an agent-worthy problem**  
   Build an agent when the task involves multiple steps, decisions, tool calls, or exceptions that would otherwise require human judgment. A simple chatbot, classifier, or single-turn LLM call is usually not an agent. Agents should be able to manage workflow execution, recognize completion, recover from errors, and hand control back to the user when necessary. [1]

2. **Start with the three core components**  
   Every agent should have:
   - **Model:** Handles reasoning and decision-making  
   - **Tools:** APIs, functions, databases, or external systems it can use  
   - **Instructions:** Define its role, workflow, constraints, and behavior [1]

3. **Prototype with the strongest suitable model**  
   Begin with a highly capable model to establish an accuracy baseline. Once the workflow works, evaluate whether smaller or faster models can handle simpler subtasks. Use stronger models for co

## 6. Role-Based Access Control (RBAC)

Everything so far assumes one kind of user: whoever asks gets whatever the router
picks. In a real deployment that's rarely true. An engineer shouldn't be able to pull
finance's 10-K numbers out of the vector store, and a finance analyst has no business
reading internal engineering docs — even though both are talking to the same agent.

RBAC puts a **permission check between the router's decision and the tool call**. The
router still reasons about *where* the answer lives; RBAC decides whether *this user*
is allowed to go there. If not, the request is rejected before any embedding, vector
search, or grounding call happens.

**This demo — 2 roles, 3 knowledge sources:**

| Knowledge source | Route label | `engineer` | `finance_analyst` |
|---|---|---|---|
| 📘 OpenAI documentation (Qdrant) | `OPENAI_QUERY` | ✅ | ✅ |
| 📗 10-K filings (Qdrant) | `10K_DOCUMENT_QUERY` | ❌ | ✅ |
| 🌐 Live internet search (SerpApi) | `INTERNET_QUERY` | ✅ | ❌ |

The two Qdrant collections and the SerpApi tool are the same ones built above — RBAC
is a layer on top, not a different pipeline.


In [22]:
# ── Users → role ─────────────────────────────────────────────────────────────
# Stand-in for a real identity provider. In production this comes from SSO/JWT
# claims or an internal users table — never a dict in the notebook.
USERS = {
    "alice": "engineer",
    "bob":   "finance_analyst",
}

# ── Roles → the route labels each role may reach ─────────────────────────────
# This is an allow-list: anything not listed here is denied by default.
ROLE_PERMISSIONS = {
    "engineer":        {"OPENAI_QUERY", "INTERNET_QUERY"},
    "finance_analyst": {"OPENAI_QUERY", "10K_DOCUMENT_QUERY"},
}

# Human-readable names, used only for clearer denial messages
SOURCE_LABELS = {
    "OPENAI_QUERY":       "OpenAI documentation",
    "10K_DOCUMENT_QUERY": "10-K financial filings",
    "INTERNET_QUERY":     "live internet search",
}


def has_access(user_id: str, action: str) -> bool:
    """True only if this user's role is explicitly allowed to use this route."""
    role = USERS.get(user_id)
    return role is not None and action in ROLE_PERMISSIONS.get(role, set())


def allowed_sources(user_id: str) -> set:
    """Every route label this user may reach — useful for constraining the router."""
    return ROLE_PERMISSIONS.get(USERS.get(user_id), set())


print("alice  (engineer)        →", allowed_sources("alice"))
print("bob    (finance_analyst) →", allowed_sources("bob"))
print("carol  (unknown user)    →", allowed_sources("carol"))

alice  (engineer)        → {'OPENAI_QUERY', 'INTERNET_QUERY'}
bob    (finance_analyst) → {'10K_DOCUMENT_QUERY', 'OPENAI_QUERY'}
carol  (unknown user)    → set()


In [23]:
def secure_agentic_rag(user_id: str, user_query: str):
    """
    The same agentic RAG loop as above, with one addition: after the router picks a
    route, the user's role must permit that route before the tool is called.

    Order of operations:
        1. Identify the user  → unknown users are rejected immediately
        2. Route the query    → router decides which knowledge source fits
        3. RBAC check         → role allowed to use that source? deny if not
        4. Retrieve + answer  → only ever reached by an authorized request

    Args:
        user_id (str): Who is asking (looked up in USERS).
        user_query (str): The question.

    Returns:
        str: The answer, or a denial message.
    """
    CYAN, GREY, RED, GREEN, BOLD, RESET = (
        "\033[96m", "\033[90m", "\033[91m", "\033[92m", "\033[1m", "\033[0m"
    )

    role = USERS.get(user_id)
    print(f"{BOLD}{CYAN}👤 User:{RESET} {user_id}  (role: {role or 'UNKNOWN'})")
    print(f"{BOLD}{CYAN}❓ Query:{RESET} {user_query}\n")

    # Step 1 — unknown identity is denied before anything else runs
    if role is None:
        print(f"{RED}🚫 ACCESS DENIED{RESET} — unknown user '{user_id}'.\n")
        return f"🚫 Access denied: unknown user '{user_id}'."

    # Step 2 — the router still does the reasoning about where the answer lives
    try:
        decision = route_query(user_query)
    except Exception as route_err:
        return f"Routing error: {route_err}"

    action = decision.get("action")
    reason = decision.get("reason")
    print(f"{GREY}📍 Selected Route: {action}")
    print(f"📝 Reason: {reason}{RESET}\n")

    # Step 3 — the gate. Nothing is embedded, searched, or grounded past this point
    #          unless the role is permitted to use the chosen source.
    if not has_access(user_id, action):
        source = SOURCE_LABELS.get(action, action)
        print(f"{RED}🚫 ACCESS DENIED{RESET} — role '{role}' may not query {source}.\n")
        return (
            f"🚫 Access denied: your role ('{role}') does not have permission to "
            f"query {source}."
        )

    print(f"{GREEN}✅ Access granted{RESET} — processing...\n")

    # Step 4 — identical to agentic_rag() from Section 5
    try:
        route_function = routes.get(action)
        if not route_function:
            return f"Unsupported action: {action}"
        if action in ["OPENAI_QUERY", "10K_DOCUMENT_QUERY"]:
            result = asyncio.run(route_function(user_query, action))
        else:
            result = route_function(user_query, action)
    except Exception as exec_err:
        result = f"Execution error: {exec_err}"

    print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n")
    print(f"{result}\n")
    return result

### Demo — same question, different roles

Each pair below sends the *identical* query as `alice` (engineer) and `bob`
(finance_analyst). The router makes the same decision both times — only the
permission check differs.


In [24]:
print("=" * 70)
print("1) alice (engineer) asks about the 10-K — FINANCE-ONLY → DENIED")
print("=" * 70)
secure_agentic_rag("alice", "what was uber revenue in 2021?")

1) alice (engineer) asks about the 10-K — FINANCE-ONLY → DENIED
👤 User: alice  (role: engineer)
❓ Query: what was uber revenue in 2021?



📍 Selected Route: 10K_DOCUMENT_QUERY
📝 Reason: Asks for Uber's annual financial revenue

🚫 ACCESS DENIED — role 'engineer' may not query 10-K financial filings.



"🚫 Access denied: your role ('engineer') does not have permission to query 10-K financial filings."

In [25]:
print("=" * 70)
print("2) bob (finance_analyst) asks the same question → ALLOWED")
print("=" * 70)
secure_agentic_rag("bob", "what was uber revenue in 2021?")

2) bob (finance_analyst) asks the same question → ALLOWED
👤 User: bob  (role: finance_analyst)
❓ Query: what was uber revenue in 2021?



📍 Selected Route: 10K_DOCUMENT_QUERY
📝 Reason: Asks for Uber's annual revenue from a 10-K financial report

✅ Access granted — processing...



🤖 BOT RESPONSE:

Uber’s revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**), up 57% from 2020.[1]



'Uber’s revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**), up 57% from 2020.[1]'

In [26]:
print("=" * 70)
print("3) bob (finance_analyst) asks for live web results — ENGINEER-ONLY → DENIED")
print("=" * 70)
secure_agentic_rag("bob", "List me down new LLMs in 2025")

3) bob (finance_analyst) asks for live web results — ENGINEER-ONLY → DENIED
👤 User: bob  (role: finance_analyst)
❓ Query: List me down new LLMs in 2025



📍 Selected Route: INTERNET_QUERY
📝 Reason: Requests a current list of newly released large language models.

🚫 ACCESS DENIED — role 'finance_analyst' may not query live internet search.



"🚫 Access denied: your role ('finance_analyst') does not have permission to query live internet search."

In [27]:
print("=" * 70)
print("4) alice (engineer) asks about OpenAI docs — SHARED → ALLOWED")
print("=" * 70)
secure_agentic_rag("alice", "best ways to build Agents")

4) alice (engineer) asks about OpenAI docs — SHARED → ALLOWED
👤 User: alice  (role: engineer)
❓ Query: best ways to build Agents



📍 Selected Route: OPENAI_QUERY
📝 Reason: Building agents relates to OpenAI agent documentation

✅ Access granted — processing...



🤖 BOT RESPONSE:

## Best ways to build agents

1. **Start with the right use case**  
   Build an agent when the task involves multiple steps, decisions, tool calls, or exceptions—not for simple chat, classification, or single-turn generation. Agents are most valuable when they can complete a workflow independently on the user’s behalf. [1]

2. **Define the core components clearly**  
   A reliable agent typically has three parts:  
   - **Model:** handles reasoning and decisions  
   - **Tools:** APIs or functions used to retrieve information or take actions  
   - **Instructions:** define behavior, workflow steps, and constraints [2]

3. **Prototype with the strongest suitable model**  
   Use a highly capable model initially to establish an accuracy baseline. Then evaluate whether smaller, faster, and less expensive models can handle individual subtasks without reducing quality. [3]

4. **Give agents focused, well-designed tools**  
   Tools should have clear names, descriptions, in

'## Best ways to build agents\n\n1. **Start with the right use case**  \n   Build an agent when the task involves multiple steps, decisions, tool calls, or exceptions—not for simple chat, classification, or single-turn generation. Agents are most valuable when they can complete a workflow independently on the user’s behalf. [1]\n\n2. **Define the core components clearly**  \n   A reliable agent typically has three parts:  \n   - **Model:** handles reasoning and decisions  \n   - **Tools:** APIs or functions used to retrieve information or take actions  \n   - **Instructions:** define behavior, workflow steps, and constraints [2]\n\n3. **Prototype with the strongest suitable model**  \n   Use a highly capable model initially to establish an accuracy baseline. Then evaluate whether smaller, faster, and less expensive models can handle individual subtasks without reducing quality. [3]\n\n4. **Give agents focused, well-designed tools**  \n   Tools should have clear names, descriptions, inp

**Where this is still weak — and how you'd harden it:**

- **The router runs before the check.** One LLM call is spent classifying a query the
  user may not be allowed to ask. That's cheap and leaks nothing, but you can do
  better: pass `allowed_sources(user_id)` into the router prompt so it only ever
  chooses from routes the role can reach, and deny anything that falls outside.
- **This gates whole sources, not chunks.** When one collection mixes content that
  different roles may only *partially* see, push the check into the vector store with
  **payload-based filters** (Qdrant supports this natively) so restricted chunks never
  enter the retrieved context in the first place. You'll do exactly this, at file
  granularity, in `003. Agentic Router_semantic_caching_rbac.ipynb`.
- **Caching and RBAC interact badly if you're careless.** A shared semantic cache
  keyed only on the question will happily serve `bob`'s finance answer to `alice`.
  Any cache sitting behind an access check must be partitioned by role (or by the
  permitted source set) — think about this before you add one.
- **Every check is an audit point.** Log who asked for what and whether it was
  allowed; that trail is what makes the system defensible in a regulated environment.


# Assignment

**Required:** Part 1 — sub-query division. This is the graded piece for this notebook.

**Bonus (optional, ungraded):** RBAC with a semantic cache. It extends Section 6 and is a
useful warm-up for **ARGUS**, where you build multi-source retrieval with a real caching
layer and have to report cost with and without the cache.

| | Task | Status | Builds on |
|---|---|---|---|
| **Part 1** | Sub-query division | **Required** | Sections 2 & 5 |
| **Bonus** | RBAC + semantic cache, without cross-role leakage | Optional | Section 6 |

**Deliverable:** this notebook, run end to end, with Part 1 implemented in the stub cell.
If you take the bonus, include it in the same notebook with the self-check passing.


---

## Part 1 — Sub-query division

Right now a compound question is treated as one search. Ask *"What was Uber's revenue
in 2021 and what was Lyft's in 2024?"* and the router picks a single route and fires a
single retrieval — so you get a partial answer, or a muddled one.

Your job: break compound queries into focused sub-queries, run each one through the
full agentic pipeline independently, then compose a single coherent answer.

**Requirements**

1. Write `agentic_rag_multi(user_query)` that:
   - calls `sub_queries()` to split the query (reference implementation below),
   - **routes each sub-query separately** — they may legitimately land on different
     sources (one on `10K_DOCUMENT_QUERY`, another on `INTERNET_QUERY`),
   - collects the per-sub-query answers and synthesises **one** final response,
   - preserves citations from each sub-answer in the composed output.
2. Handle the single-question case without regression — one question in, one route,
   no extra LLM calls beyond the split.
3. Parse the model's JSON defensively. `sub_queries()` returns a *string*; it can come
   back wrapped in prose or a code fence. Don't let a malformed split crash the agent —
   fall back to treating the input as one query.

**Check yourself against these**

| Query | Expected behaviour |
|---|---|
| `"what was uber revenue in 2021?"` | 1 sub-query, 1 route, same as `agentic_rag()` |
| `"what was lyft revenue in 2021 and what was uber revenue in 2021"` | 2 sub-queries, both `10K_DOCUMENT_QUERY` |
| `"what was uber's 2021 revenue and what are the newest LLMs?"` | 2 sub-queries, **different** routes |

**Stretch:** run the sub-queries concurrently with `asyncio.gather` instead of in
sequence, and compare wall-clock time.


In [28]:
#Reference Code for sub query division (For Guidance Only)

def sub_queries(user_query):
  sub_queries_prompt= f"""
  You are a query router. If the input contains multiple distinct questions, break it into sub-questions. Otherwise, keep it as one. Return a JSON object like:

  {{
      "subQuestions": ["..."]
  }}


  Query: "{user_query}"
  Output:
  """
  response = openaiclient.chat.completions.create(
        model="gpt-5.6-luna",
        messages=[
            {"role": "system", "content": sub_queries_prompt},
        ]
    )
  return response.choices[0].message.content


In [29]:
print(sub_queries("what was lyft revenue in 2021 and what was uber revenue in 2021"))

{"subQuestions":["What was Lyft's revenue in 2021?","What was Uber's revenue in 2021?"]}


In [30]:
# ── Part 1: your implementation ──────────────────────────────────────────────
#
# Design notes (Kurt Lozier, Cohort 2):
#   * Every sub-query goes through the SAME router + route table the single-query
#     agent uses (route_query + routes). Nothing above this cell is modified.
#   * Citations: each sub-answer carries [1][2]… local to its own retrieved
#     context. Before composing, I renumber them to [i.n] (sub-query i, source n)
#     so two sub-answers never collide, then ask the model to keep the markers
#     verbatim. A "Sources" footer maps every [i.n] back to the snippet that
#     backs it, so the trail survives synthesis.
#   * Single question ⇒ no compose call. LLM calls = split + route + answer,
#     exactly one more than agentic_rag() (the split), as the brief requires.
#   * Stretch: concurrent=True runs sub-queries with asyncio.gather. The router
#     and the internet tool are blocking calls, so they run in worker threads
#     via asyncio.to_thread; the Qdrant path is already async.

import asyncio
import json
import re
import time

import nest_asyncio
nest_asyncio.apply()          # lets asyncio.run() work inside Jupyter's loop

MODEL_NAME = "gpt-5.6-luna"   # same model the rest of the notebook uses
MAX_SUB_QUERIES = 5           # guard against a runaway split

_QDRANT_COLLECTIONS = {
    "OPENAI_QUERY": "opnai_data",
    "10K_DOCUMENT_QUERY": "10k_data",
}


def parse_sub_queries(raw: str, fallback: str) -> list:
    """
    Defensive parser for sub_queries() output.

    Accepts: bare JSON, JSON inside a ```json fence, JSON surrounded by prose.
    Returns a list of 1..MAX_SUB_QUERIES non-empty strings; on ANY problem
    returns [fallback] so a bad split degrades to single-query behaviour
    instead of crashing the agent.
    """
    if not isinstance(raw, str) or not raw.strip():
        return [fallback]
    text = raw.strip()
    # strip a code fence if present
    fence = re.search(r"```(?:json)?\s*(.*?)```", text, flags=re.S | re.I)
    if fence:
        text = fence.group(1).strip()
    # take the outermost {...} in case the model wrapped it in prose
    brace = re.search(r"\{.*\}", text, flags=re.S)
    if brace:
        text = brace.group(0)
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        return [fallback]
    subs = data.get("subQuestions") if isinstance(data, dict) else None
    if not isinstance(subs, list):
        return [fallback]
    cleaned = [s.strip() for s in subs if isinstance(s, str) and s.strip()]
    if not cleaned:
        return [fallback]
    return cleaned[:MAX_SUB_QUERIES]


def _renumber_citations(answer: str, idx: int) -> str:
    """[3] → [2.3] for sub-query 2, so citations from different sub-answers never collide."""
    return re.sub(r"\[(\d+)\]", lambda m: f"[{idx}.{m.group(1)}]", answer)


async def _answer_one(sub_query: str, idx: int) -> dict:
    """
    Route ONE sub-query and answer it. Returns a dict with the route decision,
    the answer (citations renumbered to [idx.n]) and the raw sources.

    Mirrors retrieve_and_response() for Qdrant routes but keeps the retrieved
    snippets so the composed answer can print a sources footer. Internet route
    delegates to the notebook's get_internet_content() unchanged.
    """
    try:
        decision = await asyncio.to_thread(route_query, sub_query)
    except Exception as e:
        return {"idx": idx, "query": sub_query, "action": None, "reason": None,
                "answer": f"Routing error: {e}", "sources": []}

    action = decision.get("action")
    reason = decision.get("reason")
    sources = []

    try:
        if action in _QDRANT_COLLECTIONS:
            vec = get_text_embeddings(sub_query)
            hits = await client.query_points(
                collection_name=_QDRANT_COLLECTIONS[action], query=vec, limit=3
            )
            sources = [p.payload.get("content", "") for p in hits.points]
            context = [f"[{i + 1}] {c}" for i, c in enumerate(sources)]
            answer = await asyncio.to_thread(rag_formatted_response, sub_query, context)
        elif action in routes:                          # INTERNET_QUERY
            answer = await asyncio.to_thread(routes[action], sub_query, action)
            # get_internet_content() already numbers results "[n] title / snippet /
            # Source: url" — lift title + url so [idx.n] resolves in the footer
            sources = [f"{t.strip()} — {u}" for t, u in
                       re.findall(r"\[\d+\] (.*?)\n\s+.*?\n\s+Source: (\S+)", str(answer), flags=re.S)]
            if not sources:
                sources = ["(live web search returned no linked results)"]
        else:
            answer = f"Unsupported action: {action}"
    except Exception as e:
        answer = f"Execution error: {e}"

    return {"idx": idx, "query": sub_query, "action": action, "reason": reason,
            "answer": _renumber_citations(str(answer), idx), "sources": sources}


def _compose(user_query: str, parts: list) -> str:
    """One LLM call that merges the sub-answers into a single coherent response."""
    blocks = "\n\n".join(
        f"Sub-question {p['idx']} ({p['action']}): {p['query']}\nAnswer {p['idx']}:\n{p['answer']}"
        for p in parts
    )
    prompt = f"""You are composing a final answer to a compound user question from
several independently researched sub-answers.

Rules:
- Answer the ORIGINAL question in one coherent response.
- Keep every citation marker of the form [i.n] EXACTLY as written; do not
  renumber, merge, or drop them. Do not invent new ones.
- If a sub-answer says it could not find something, say so plainly.

Original question: {user_query}

{blocks}
"""
    resp = openaiclient.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "system", "content": prompt}],
    )
    return resp.choices[0].message.content


def _sources_footer(parts: list) -> str:
    lines = ["", "── Sources ──"]
    for p in parts:
        lines.append(f"Sub-query {p['idx']} → {p['action']}: {p['query']}")
        for n, s in enumerate(p["sources"], start=1):
            snippet = re.sub(r"\s+", " ", s)[:160]
            lines.append(f"  [{p['idx']}.{n}] {snippet}…")
    return "\n".join(lines)


def agentic_rag_multi(user_query: str, concurrent: bool = True) -> str:
    """
    Split a compound query, run each sub-query through the agentic pipeline,
    and synthesise one final answer.

    Args:
        user_query (str): Possibly compound question.
        concurrent (bool): run sub-queries with asyncio.gather (stretch goal)
                           or sequentially, for the wall-clock comparison.

    Returns:
        str: A single composed answer covering every sub-question.
    """
    CYAN, GREY, BOLD, RESET = "\033[96m", "\033[90m", "\033[1m", "\033[0m"
    print(f"{BOLD}{CYAN}👤 User Query:{RESET} {user_query}\n")
    t0 = time.perf_counter()

    # 1. Split — parse defensively, fall back to one query
    try:
        raw = sub_queries(user_query)
    except Exception as e:
        print(f"{GREY}⚠️ split failed ({e}); treating input as one query{RESET}")
        raw = ""
    subs = parse_sub_queries(raw, fallback=user_query)
    print(f"{GREY}🔀 {len(subs)} sub-query(ies): {subs}{RESET}\n")

    # 2. Route + answer each sub-query
    async def _run_all():
        if concurrent:
            return await asyncio.gather(*(_answer_one(q, i + 1) for i, q in enumerate(subs)))
        return [await _answer_one(q, i + 1) for i, q in enumerate(subs)]

    parts = asyncio.run(_run_all())
    for p in parts:
        print(f"{GREY}📍 Sub-query {p['idx']} → {p['action']}  ({p['reason']}){RESET}")

    # 3. Compose — skipped for a single question (no extra LLM call)
    if len(parts) == 1:
        final = re.sub(r"\[1\.(\d+)\]", r"[\1]", parts[0]["answer"])  # back to [n]
    else:
        final = _compose(user_query, parts)
    final += _sources_footer(parts)

    elapsed = time.perf_counter() - t0
    mode = "concurrent" if concurrent else "sequential"
    print(f"\n{BOLD}{CYAN}🤖 BOT RESPONSE{RESET} {GREY}({mode}, {elapsed:.1f}s){RESET}\n")
    print(final, "\n")
    return final


**Parser robustness** — no API calls; proves a malformed split degrades to one query instead of crashing.


In [31]:
_fb = "fallback question"
assert parse_sub_queries('{"subQuestions": ["a", "b"]}', _fb) == ["a", "b"]
assert parse_sub_queries('Sure! ```json\n{"subQuestions": ["a"]}\n```', _fb) == ["a"]
assert parse_sub_queries('Here you go: {"subQuestions": ["a", "b"]} hope it helps', _fb) == ["a", "b"]
assert parse_sub_queries('not json at all', _fb) == [_fb]
assert parse_sub_queries('{"subQuestions": []}', _fb) == [_fb]
assert parse_sub_queries('{"subQuestions": "a"}', _fb) == [_fb]
assert parse_sub_queries('', _fb) == [_fb]
assert len(parse_sub_queries(json.dumps({"subQuestions": [str(i) for i in range(20)]}), _fb)) == MAX_SUB_QUERIES
print("✅ parse_sub_queries handles fenced, prose-wrapped, malformed, and empty splits")


✅ parse_sub_queries handles fenced, prose-wrapped, malformed, and empty splits


**The three check-yourself cases** — 1 sub-query (no compose call), 2 sub-queries on the same route, 2 sub-queries on different routes.


In [32]:
# Test cases from the table above
_ = agentic_rag_multi("what was uber revenue in 2021?")
_ = agentic_rag_multi("what was lyft revenue in 2021 and what was uber revenue in 2021")
_ = agentic_rag_multi("what was uber's 2021 revenue and what are the newest LLMs?")


👤 User Query: what was uber revenue in 2021?



🔀 1 sub-query(ies): ["What was Uber's revenue in 2021?"]



📍 Sub-query 1 → 10K_DOCUMENT_QUERY  (Asks for Uber's annual financial revenue)

🤖 BOT RESPONSE (concurrent, 2.8s)

Uber’s revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**). [1][2]
── Sources ──
Sub-query 1 → 10K_DOCUMENT_QUERY: What was Uber's revenue in 2021?
  [1.1] during the years ended December 31, 2020 and 2021, respectively. Highlights for 2021 Overall Gross Bookings increased by $32.5 billion in 2021, up 56%, or 53% o…
  [1.2] Year Ended December 31, 2019 2020 2021 Revenue $ 13,000 $ 11,139 $ 17,455 Costs and expenses Cost of revenue, exclusive of depreciation and amortization shown s…
  [1.3] % change 2021 to 2020 % Change 2022 2021 2020 (in thousands, except for percentages) Revenue $ 4,095,135 $ 3,208,323 $ 2,364,681 28 % 36 % Revenue increased $88… 

👤 User Query: what was lyft revenue in 2021 and what was uber revenue in 2021



🔀 2 sub-query(ies): ["What was Lyft's revenue in 2021?", "What was Uber's revenue in 2021?"]



📍 Sub-query 1 → 10K_DOCUMENT_QUERY  (Asks for Lyft's historical financial revenue)
📍 Sub-query 2 → 10K_DOCUMENT_QUERY  (Asks about a company’s annual financial revenue)



🤖 BOT RESPONSE (concurrent, 5.3s)

In 2021:

- **Lyft revenue:** **$3.208 billion** (approximately $3,208,323,000).[1.2]
- **Uber revenue:** **$17.455 billion** (approximately $17.5 billion).[2.1][2.2]
── Sources ──
Sub-query 1 → 10K_DOCUMENT_QUERY: What was Lyft's revenue in 2021?
  [1.1] network of Light Vehicles, and Lyft Rentals, which is recognized in accordance with Accounting Standards Codification Topic 842 (“ASC 842”). The table below pre…
  [1.2] 250,344 156,025 Total revenue $ 4,095,135 $ 3,208,323 $ 2,364,681 84 Revenue from Contracts with Customers (ASC 606) The Company recognizes revenue for its ride…
  [1.3] unless the ride is accessible in the Lyft App. Revenue per Active Rider is calculated by dividing revenue for a period by Active Riders for the same period. Beg…
Sub-query 2 → 10K_DOCUMENT_QUERY: What was Uber's revenue in 2021?
  [2.1] during the years ended December 31, 2020 and 2021, respectively. Highlights for 2021 Overall Gross Bookings increased by $32.5 bill

🔀 2 sub-query(ies): ["What was Uber's revenue in 2021?", 'What are the newest large language models (LLMs)?']



Getting your response from the internet 🌐 ...


📍 Sub-query 1 → 10K_DOCUMENT_QUERY  (Asks for a company's annual financial figure)
📍 Sub-query 2 → INTERNET_QUERY  (Requests current LLM landscape beyond OpenAI-specific information)



🤖 BOT RESPONSE (concurrent, 6.7s)

Uber’s **2021 revenue was $17.455 billion**, or approximately **$17.5 billion**. [1.1][1.2]

As for the **newest large language models (LLMs)**, there is no single universally accepted ranking because models are released and updated frequently. Recent listings and tracking sources mention:

- **Amazon Nova Micro, Nova Lite, and Nova Pro**
- **Deep Think** and **Pro** models
- Current model families from **OpenAI’s GPT and o-series, Anthropic’s Claude, and Google’s models** [2.2][2.3]
- **GPT-4o**, OpenAI’s multimodal flagship released in May 2024, though it is not necessarily the newest model by September 2026 [2.4]

LLMs are AI systems trained on large amounts of text and designed to understand and generate human language. [2.1][2.5]
── Sources ──
Sub-query 1 → 10K_DOCUMENT_QUERY: What was Uber's revenue in 2021?
  [1.1] during the years ended December 31, 2020 and 2021, respectively. Highlights for 2021 Overall Gross Bookings increased by $32.5 bil

**Stretch — sequential vs concurrent.** Same compound query, run both ways; the concurrent path overlaps the router, retrieval, and answer calls of each sub-query.


In [33]:
import time as _t

_q = "what was uber's 2021 revenue and what are the newest LLMs?"
_t0 = _t.perf_counter(); agentic_rag_multi(_q, concurrent=False); seq = _t.perf_counter() - _t0
_t0 = _t.perf_counter(); agentic_rag_multi(_q, concurrent=True);  con = _t.perf_counter() - _t0
print(f"sequential: {seq:.1f}s   concurrent: {con:.1f}s   speed-up: {seq / con:.2f}×")


👤 User Query: what was uber's 2021 revenue and what are the newest LLMs?



🔀 2 sub-query(ies): ["What was Uber's revenue in 2021?", 'What are the newest large language models (LLMs)?']



Getting your response from the internet 🌐 ...
📍 Sub-query 1 → 10K_DOCUMENT_QUERY  (Asks about Uber's annual financial revenue)
📍 Sub-query 2 → INTERNET_QUERY  (Requires current information about models across providers)



🤖 BOT RESPONSE (sequential, 9.7s)

Uber reported **$17.455 billion in revenue in 2021**—approximately **$17.5 billion**. [1.1][1.2]

Large language models (LLMs) are AI systems trained on vast amounts of text to understand and generate human language. [2.1][2.5] The newest models vary by provider and release date. Examples identified in the available sources include:

- **Amazon Nova Micro, Nova Lite, and Nova Pro** [2.2]
- Newer models referred to as **Deep Think** and **Pro** [2.2]
- Current model families from **OpenAI, Anthropic, and Google**, whose releases are tracked by LLM Stats [2.3]
- **OpenAI GPT-4o**, a multimodal flagship model highlighted in one 2026 overview, though it was originally released in May 2024 [2.4]

The provided sources do not identify one definitive, universally accepted “newest LLM”; that depends on the provider and the date of comparison.
── Sources ──
Sub-query 1 → 10K_DOCUMENT_QUERY: What was Uber's revenue in 2021?
  [1.1] during the years ended Decemb

🔀 2 sub-query(ies): ["What was Uber's revenue in 2021?", 'What are the newest large language models (LLMs)?']



Getting your response from the internet 🌐 ...


📍 Sub-query 1 → 10K_DOCUMENT_QUERY  (Uber's 2021 revenue is reported in its annual filing)
📍 Sub-query 2 → INTERNET_QUERY  (Requires current information about models across providers)



🤖 BOT RESPONSE (concurrent, 7.1s)

Uber’s revenue in 2021 was **$17.455 billion**—approximately **$17.5 billion**. [1.1][1.2]

**Large language models (LLMs)** are AI systems trained on vast amounts of text that can understand and generate human language. [2.1][2.5] There is no single definitive “newest” LLM, since models are released and updated frequently. Recent examples and model families referenced by the sources include:

- **Amazon Nova Micro, Nova Lite, and Nova Pro**, along with models listed as **Deep Think** and **Pro**. [2.2]
- **OpenAI’s GPT and o-series models**, **Anthropic’s Claude models**, and **Google’s model families**, which are tracked through ongoing release updates. [2.3]
- **GPT-4o**, OpenAI’s multimodal flagship model, released in May 2024. [2.4]

Thus, the answer is **$17.455 billion in 2021 revenue**, while the newest LLM landscape is continually changing and includes models from OpenAI, Anthropic, Google, Amazon, and others.
── Sources ──
Sub-query 1 → 10K

---

## Bonus (optional) — RBAC with a semantic cache

*Not graded on its own — but this is rehearsal for graded work. **ARGUS**, the full-stack
assignment for this module, requires a real caching layer and a cost panel showing spend per
100 sources with and without it. Build the cache here, on a system you already understand,
and you'll be extending it in ARGUS rather than meeting it for the first time under a deadline.*

> **Come back to this after notebooks 002 and 003** — `002. Semantic Caching.ipynb`
> for how a FAISS cache works, and `003. Agentic Router_semantic_caching_rbac.ipynb` for `SemanticCaching`
> in `rag_helpers.py` and the file-level RBAC gate you'll be extending. You can read the spec now;
> you'll have every piece you need once those two are done.

Section 6 gates access by role. A semantic cache makes repeat questions near-instant.
Put them together naively and you build a data leak: the cache is keyed on the
*question*, so once `bob` (finance_analyst) asks about Uber's revenue, `alice`
(engineer) asks the same thing, hits the cache, and is handed finance data the RBAC
gate was supposed to deny her — without a single retrieval ever running.

Your job: add caching to `secure_agentic_rag()` so that repeat questions are fast
**and** no answer ever crosses a permission boundary.

**Requirements**

1. Build a role-aware cache. Two viable designs — pick one and justify it in a comment:
   - **Partitioned:** a separate FAISS index (or a namespace) per role, so a lookup
     can only ever see entries its own role produced.
   - **Tagged:** one index, but each entry stores the role (or the permitted source
     set) that produced it, and a hit is only honoured when it matches the caller.
2. Write `secure_agentic_rag_cached(user_id, user_query)` with this order of
   operations — it matters:
   ```
   unknown user      → DENIED   (no embedding, no cache, no LLM)
   route the query   → which source does this need?
   role not allowed  → DENIED   (still no cache lookup — a denial must not be cacheable)
   cache lookup      → HIT  → return stored answer
                     → MISS → run the pipeline, store, return
   ```
3. Return a dict, not a bare string, so the self-check can verify behaviour:
   `{"answer": str, "status": "HIT" | "MISS" | "DENIED", "role": str | None}`
4. Never cache a denial, and never cache a time-sensitive query. Reuse the
   `is_time_sensitive()` idea from `003. Agentic Router_semantic_caching_rbac.ipynb` — a stock
   price cached for an hour is a wrong answer served fast.
5. Make the leak test below pass.

**Hints**

- `SemanticCaching` in `rag_helpers.py` is a working FAISS cache — read it first.
  Partitioning it is mostly a matter of what you key and what you search.
- Think about what happens when a user's role *changes*. Should their old cache
  entries still be reachable? Write a sentence on your answer.
- Two roles share `OPENAI_QUERY`. A strictly per-role cache re-computes that answer
  once per role — correct, but wasteful. Caching per *permitted source* instead of per
  role fixes it. Trade-off worth a comment.

**Stretch:** log every request as an audit record — user, role, query, route, decision,
cache status, latency — and print a small table at the end. That table is what you'd
hand an auditor.


In [34]:
# ── Bonus: your implementation ───────────────────────────────────────────────
# Reuse USERS / ROLE_PERMISSIONS / has_access / SOURCE_LABELS from Section 6.

import numpy as np
import re as _re
import time as _time

AUDIT_LOG = []   # stretch: one record per request, printed as a table at the end


def is_time_sensitive(query: str) -> bool:
    """
    Cheap heuristic: a question whose right answer changes over time must not be
    served from cache. (Borrowed idea from notebook 003.) The INTERNET_QUERY route
    is additionally never cached at all — live search is time-sensitive by nature.
    """
    pattern = r"\b(today|now|current(ly)?|latest|newest|recent(ly)?|this (week|month|year)|" \
              r"right now|live|breaking|stock price|price of|weather)\b"
    return _re.search(pattern, query, flags=_re.I) is not None


class RoleAwareSemanticCache:
    """
    A semantic cache that cannot serve an answer across a permission boundary.

    Design choice (partitioned vs. tagged): TAGGED BY SOURCE.
      Every entry is tagged with the ROUTE (knowledge source) that produced it,
      not the role. A lookup only considers entries whose source the caller's
      role is permitted to read, and secure_agentic_rag_cached() additionally
      narrows the search to the single route the query needs.
      Why not per-role partitions? Both roles may read OpenAI docs; per-role
      caching would compute that answer once per role. Tagging by source lets
      alice and bob share OPENAI_QUERY rows while 10K rows stay invisible to
      alice — the permission check, not the cache layout, is what protects them.
      Role change: entries stay valid. Permissions are re-evaluated on every
      request BEFORE the cache is consulted, so a user promoted or demoted simply
      sees a different slice of the same cache; nothing needs flushing.
      Index: numpy cosine similarity over the notebook's nomic embeddings. The
      course pattern (SemanticCaching in rag_helpers.py) uses FAISS; at notebook
      scale a brute-force matrix product is the same search without the extra
      dependency. Swap in faiss.IndexFlatIP for >10k rows.
    """

    def __init__(self, threshold: float = 0.2):
        # threshold is a cosine DISTANCE (1 - cosine similarity), matching the
        # FAISS-distance convention of the course cache: smaller = closer.
        self.threshold = threshold
        self._vecs = []      # list of unit-norm np arrays
        self._rows = []      # list of dicts: question, answer, action

    @staticmethod
    def _unit(embedding) -> np.ndarray:
        v = np.asarray(embedding, dtype=np.float32).reshape(-1)
        n = np.linalg.norm(v)
        return v / n if n else v

    def check(self, user_id: str, question: str, action: str | None = None, embedding=None):
        """
        Return (hit: bool, answer: str | None, embedding, similarity | None).
        Only entries the caller's role may read are eligible; if `action` is
        given, only entries from that route are eligible (strict partition).
        """
        if embedding is None:
            embedding = get_text_embeddings(question)
        q = self._unit(embedding)
        allowed = allowed_sources(user_id)
        eligible = [
            i for i, r in enumerate(self._rows)
            if r["action"] in allowed and (action is None or r["action"] == action)
        ]
        if not eligible:
            return False, None, embedding, None
        mat = np.stack([self._vecs[i] for i in eligible])
        sims = mat @ q
        best = int(np.argmax(sims))
        sim = float(sims[best])
        if (1.0 - sim) <= self.threshold:
            return True, self._rows[eligible[best]]["answer"], embedding, sim
        return False, None, embedding, sim

    def add(self, user_id: str, question: str, answer: str, embedding, action: str | None = None):
        """Store an answer tagged with the source that produced it (never a denial)."""
        if action is None:
            raise ValueError("action (route label) is required to tag a cache entry")
        if not has_access(user_id, action):
            return  # belt and braces: never store what the caller could not read
        self._vecs.append(self._unit(embedding))
        self._rows.append({"question": question, "answer": answer, "action": action,
                           "added_by": user_id})

    def __len__(self):
        return len(self._rows)


def _run_route_sync(user_query: str, action: str) -> str:
    """Step 4 of secure_agentic_rag(), factored out so the cached path can reuse it."""
    route_function = routes.get(action)
    if not route_function:
        return f"Unsupported action: {action}"
    if action in ("OPENAI_QUERY", "10K_DOCUMENT_QUERY"):
        return asyncio.run(route_function(user_query, action))
    return route_function(user_query, action)


def secure_agentic_rag_cached(user_id: str, user_query: str, cache) -> dict:
    """
    RBAC-gated agentic RAG with a role-aware semantic cache.

    Order: identity → route → permission → cache → pipeline.

    Returns:
        dict: {"answer": str, "status": "HIT" | "MISS" | "DENIED", "role": str | None}
              (+ route, similarity, latency_ms for the audit table)
    """
    t0 = _time.perf_counter()
    role = USERS.get(user_id)
    action = None
    sim = None

    def _finish(answer, status):
        rec = {"user": user_id, "role": role, "query": user_query[:48], "route": action,
               "status": status, "sim": None if sim is None else round(sim, 3),
               "ms": round((_time.perf_counter() - t0) * 1000)}
        AUDIT_LOG.append(rec)
        print(f"  {status:<6} {user_id:<6} {str(role):<16} {str(action):<19} "
              f"sim={rec['sim']}  {rec['ms']}ms")
        return {"answer": answer, "status": status, "role": role,
                "route": action, "similarity": sim, "latency_ms": rec["ms"]}

    # 1. identity — unknown user: no embedding, no cache, no LLM
    if role is None:
        return _finish(f"🚫 Access denied: unknown user '{user_id}'.", "DENIED")

    # 2. route — the router reasons about where the answer lives
    try:
        action = route_query(user_query).get("action")
    except Exception as e:
        return _finish(f"Routing error: {e}", "DENIED")

    # 3. permission — a denial must never reach the cache, so it can't be cached
    if not has_access(user_id, action):
        return _finish(
            f"🚫 Access denied: your role ('{role}') does not have permission to "
            f"query {SOURCE_LABELS.get(action, action)}.", "DENIED")

    # 4. cache — restricted to this route AND this role's permitted sources
    cacheable = action != "INTERNET_QUERY" and not is_time_sensitive(user_query)
    embedding = None
    if cacheable:
        hit, answer, embedding, sim = cache.check(user_id, user_query, action=action)
        if hit:
            return _finish(answer, "HIT")

    # 5. pipeline — identical to secure_agentic_rag() step 4, then store
    answer = _run_route_sync(user_query, action)
    if cacheable:
        cache.add(user_id, user_query, answer, embedding, action=action)
    return _finish(answer, "MISS")


def print_audit_table():
    """Stretch: the table you'd hand an auditor."""
    hdr = f"{'#':>2}  {'status':<6} {'user':<6} {'role':<16} {'route':<19} {'sim':>6} {'ms':>6}  query"
    print(hdr); print("-" * len(hdr))
    for i, r in enumerate(AUDIT_LOG, 1):
        print(f"{i:>2}  {r['status']:<6} {r['user']:<6} {str(r['role']):<16} "
              f"{str(r['route']):<19} {str(r['sim']):>6} {r['ms']:>6}  {r['query']}")


In [35]:
# ── Bonus: self-check — this must pass ───────────────────────────────────────
# It asserts behaviour, not wording, so your answer text can be anything.

def run_self_check():
    cache = RoleAwareSemanticCache()
    q_fin = "what was uber revenue in 2021?"
    q_doc = "how do I build an agent with the OpenAI Agents SDK?"

    # 1. bob may read financials — first ask is a MISS
    r = secure_agentic_rag_cached("bob", q_fin, cache)
    assert r["status"] == "MISS", f"expected MISS, got {r['status']}"

    # 2. bob asks again — served from cache
    r = secure_agentic_rag_cached("bob", q_fin, cache)
    assert r["status"] == "HIT", f"expected HIT, got {r['status']}"

    # 3. THE LEAK TEST — alice must be denied, never served bob's cached answer
    r = secure_agentic_rag_cached("alice", q_fin, cache)
    assert r["status"] == "DENIED", f"LEAK: alice got {r['status']} on finance data"

    # 4. a near-paraphrase must also be denied, not semantically matched into bob's rows
    r = secure_agentic_rag_cached("alice", "how much revenue did Uber make in 2021?", cache)
    assert r["status"] == "DENIED", f"LEAK: alice got {r['status']} via paraphrase"

    # 5. unknown users are rejected outright
    r = secure_agentic_rag_cached("carol", q_doc, cache)
    assert r["status"] == "DENIED", f"expected DENIED for unknown user, got {r['status']}"

    # 6. a shared source still caches normally within a role
    assert secure_agentic_rag_cached("alice", q_doc, cache)["status"] == "MISS"
    assert secure_agentic_rag_cached("alice", q_doc, cache)["status"] == "HIT"

    print("✅ All checks passed — cache is fast and does not leak across roles.")


# run_self_check()

**Run the leak test** and print the audit trail (stretch).


In [36]:
AUDIT_LOG.clear()
run_self_check()
print()
print_audit_table()


  MISS   bob    finance_analyst  10K_DOCUMENT_QUERY  sim=None  2944ms


  HIT    bob    finance_analyst  10K_DOCUMENT_QUERY  sim=1.0  1227ms


  DENIED alice  engineer         10K_DOCUMENT_QUERY  sim=None  1378ms


  DENIED alice  engineer         10K_DOCUMENT_QUERY  sim=None  1041ms
  DENIED carol  None             None                sim=None  0ms


  MISS   alice  engineer         OPENAI_QUERY        sim=None  9352ms


  HIT    alice  engineer         OPENAI_QUERY        sim=1.0  910ms
✅ All checks passed — cache is fast and does not leak across roles.

 #  status user   role             route                  sim     ms  query
---------------------------------------------------------------------------
 1  MISS   bob    finance_analyst  10K_DOCUMENT_QUERY    None   2944  what was uber revenue in 2021?
 2  HIT    bob    finance_analyst  10K_DOCUMENT_QUERY     1.0   1227  what was uber revenue in 2021?
 3  DENIED alice  engineer         10K_DOCUMENT_QUERY    None   1378  what was uber revenue in 2021?
 4  DENIED alice  engineer         10K_DOCUMENT_QUERY    None   1041  how much revenue did Uber make in 2021?
 5  DENIED carol  None             None                  None      0  how do I build an agent with the OpenAI Agents S
 6  MISS   alice  engineer         OPENAI_QUERY          None   9352  how do I build an agent with the OpenAI Agents S
 7  HIT    alice  engineer         OPENAI_QUERY           1.